# Wave exposure pipeline

Runs the full classification pipeline using the `waves` package.

In [1]:
import geopandas as gpd
import waves
from waves.paths import SRC, CLF, SIE, VRAW, FIN


## 1. Prepare raster data

In [ ]:
waves.prepare.bolge_model_data()


## 2. Inspect NiN 3 classification table

In [5]:
waves.classify.CLASSES


,class_int,trinn,navn_no,navn_en,swm_lower,swm_upper
0,1,0,minimal vannforstyrrelsesintensitet,still water,NaN,1200.0
1,2,a,svært beskyttet,very sheltered,1200.0,4000.0
2,3,b,temmelig beskyttet,moderately sheltered,4000.0,10000.0
3,4,c,litt beskyttet,slightly sheltered,10000.0,50000.0
4,5,d,svakt eksponert,weakly sheltered,50000.0,100000.0
5,6,e,litt eksponert,slightly exposed,100000.0,500000.0
6,7,f,temmelig eksponert,moderately exposed,500000.0,1000000.0
7,8,g,svært eksponert,very exposed,1000000.0,2000000.0
8,9,h,ekstremt eksponert,extremely exposed,2000000.0,4000000.0
9,10,y,disruptivt eksponert,disruptively exposed,4000000.0,NaN


## 3. Classify, sieve, and polygonize

In [ ]:
waves.classify.reclassify_raster(SRC, CLF)
print(f"Classified raster saved: {CLF}")


In [ ]:
waves.classify.sieve_filter(CLF, SIE, threshold=4)
print(f"Sieved raster saved: {SIE}")


In [2]:
waves.smooth.vectorize_raster(SIE, VRAW)
print(f"Vectorized polygons saved: {VRAW}")

Vectorizing /home/kim/work/eswm-bolgemodell/niva/EswmRaster_classified_sieved.tif (EPSG:32633) …
Importing raster into GRASS …


Importing raster map <raster>...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100


Vectorizing (r.to.vect -s) …


Extracting areas...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100
Writing areas...
   0   4   8  12  16  20  24  28  32  36  40  44  48  52  56  60  64  68  72  76  80  84  88  92  96 100
Building topology for vector map <vector@PERMANENT>...
Registering primitives...
Building areas...    300     400     500     600     700     800     900    1000    1100    1200    1300    1400    1500    1600    1700    1800    1900    2000    2100    2200    2300    2400    2500    2600    2700    2800    2900    3000    3100    3200    3300    3400    3500    3600    3700    3800    3900    4000    4100    4200    4300    4400    4500    4600    4700    4800    4900    5000    5100    5200    5300    5400    5500    5600    5700    5800    5900    6000    6100    6200    6300    6400    6500    6600
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  5

Generalizing with Douglas (threshold=12.0 m) …


Copying features...
   2   5   8  11  14  17  20  23  26  29  32  35  38  41  44  47  50  53  56  59  62  65  68  71  74  77  80  83  86  89  92  95  98 100
Building topology for vector map <vector_smooth@PERMANENT>...
Registering primitives...
Building areas...    300     400     500     600     700     800     900    1000    1100    1200    1300    1400    1500    1600    1700    1800    1900    2000    2100    2200    2300    2400    2500    2600    2700    2800    2900    3000    3100    3200    3300    3400    3500    3600    3700    3800    3900    4000    4100    4200    4300    4400    4500    4600    4700    4800    4900    5000    5100    5200    5300    5400    5500    5600    5700    5800    5900    6000    6100    6200    6300    6400    6500    6600
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  52  54  56  58  60  62  64  66  68  70  72  74  76  78  80  82  84  86  88  90  92  94  96  98 100
Attaching islands...


Exporting to /home/kim/work/eswm-bolgemodell/niva/Eswm_classified_raw.gpkg …
Done → /home/kim/work/eswm-bolgemodell/niva/Eswm_classified_raw.gpkg
Vectorized polygons saved: /home/kim/work/eswm-bolgemodell/niva/Eswm_classified_raw.gpkg


## 4. Subtract land

Load a previously saved checkpoint, or run `subtract_land` to (re)compute it.

In [ ]:
waves.prepare.subtract_land()



## 5. Add NiN class attributes

In [4]:
gdf = gpd.read_file("gdf_grunnlinje.gpkg")
gdf = waves.classify.add_class_attributes(gdf)
gdf.head()


,class_int,geometry,trinn,navn_no,navn_en
0,6,"MULTIPOLYGON (((56516.631 6449106.561, 56516.6...",e,litt eksponert,slightly exposed
1,6,"MULTIPOLYGON (((56541.631 6449131.561, 56541.6...",e,litt eksponert,slightly exposed
2,7,"MULTIPOLYGON (((56541.631 6449156.561, 56541.6...",f,temmelig eksponert,moderately exposed
3,5,"MULTIPOLYGON (((56759 6449181, 56757 6449178, ...",d,svakt eksponert,weakly sheltered
4,5,"MULTIPOLYGON (((56688 6449181.561, 56688 64491...",d,svakt eksponert,weakly sheltered


## 6. Save final output

In [ ]:
gdf.to_file("test.gpkg", driver="GPKG")
